In [47]:
import pandas as pd
import numpy as np
import time
import joblib
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score, f1_score, accuracy_score, classification_report

In [48]:
X_train = pd.read_csv("dataset/processed/X_train_featured.csv")
X_test = pd.read_csv("dataset/processed/X_test_featured.csv")
y_train = pd.read_csv("dataset/processed/y_train.csv")
y_test = pd.read_csv("dataset/processed/y_test.csv")

print("\n" + "="*40)
print("DATA SHAPE RESULTS")
print("="*40)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print("-" * 40)
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")


DATA SHAPE RESULTS
X_train shape: (2492096, 30)
y_train shape: (2492096, 1)
----------------------------------------
X_test shape:  (623050, 30)
y_test shape:  (623050, 1)


In [49]:
# Table to store results
results_table = []

def evaluate_regressor(model, X_train, y_train, X_test, y_test, model_name):
    # 1. PREDICTION (Predict continuous values)
    y_train_pred_raw = model.predict(X_train)
    y_test_pred_raw = model.predict(X_test)
    
    # 2. PROCESSING (Clip values between 1-4 and round for classification metrics)
    y_train_pred_class = np.round(np.clip(y_train_pred_raw, 1, 4)).astype(int)
    y_test_pred_class = np.round(np.clip(y_test_pred_raw, 1, 4)).astype(int)
    
    # 3. COMPUTE METRICS
    
    # A. Regression Metrics
    # Train
    train_mse = mean_squared_error(y_train, y_train_pred_raw)
    train_rmse = np.sqrt(train_mse)
    train_r2 = r2_score(y_train, y_train_pred_raw)
    
    # Test
    test_mse = mean_squared_error(y_test, y_test_pred_raw)
    test_rmse = np.sqrt(test_mse)
    test_r2 = r2_score(y_test, y_test_pred_raw)
    
    # B. Classification Metrics
    # Train
    train_f1 = f1_score(y_train, y_train_pred_class, average='weighted')
    train_acc = accuracy_score(y_train, y_train_pred_class)
    
    # Test
    test_f1 = f1_score(y_test, y_test_pred_class, average='weighted')
    test_acc = accuracy_score(y_test, y_test_pred_class)
    
    # 4. PRINT RESULTS
    print(f"✅ {model_name} Results:")
    print(f"   [TRAIN] MSE: {train_mse:.4f} | RMSE: {train_rmse:.4f} | Accuracy: {train_acc:.4f} | F1: {train_f1:.4f} | R2: {train_r2:.4f}")
    print(f"   [TEST] MSE: {test_mse:.4f} | RMSE: {test_rmse:.4f} | Accuracy: {test_acc:.4f} | F1: {test_f1:.4f} | R2: {test_r2:.4f}")
    print(f"   -> Gap (Overfitting): {train_f1 - test_f1:.4f} (F1 diff)")
    print("-" * 60)
    
    # 5. RETURN DICTIONARY
    return {
        'Model': model_name,
        'RMSE': test_rmse,
        'MSE': test_mse,
        'F1': test_f1,
        'Accuracy': test_acc,
        'R2': test_r2,
        'Object': model
    }

### `evaluate_regressor` Function

This function evaluates a regression model using both **regression** and **classification metrics**.

## Steps

1. **Prediction**  
   Generate raw predictions on training and test datasets.

2. **Processing**  
   Clip predictions to the range `[1, 4]` and round them to compute classification-like metrics.

3. **Metrics Computation**  
   - **Regression Metrics:** MSE, RMSE, R²  
   - **Classification Metrics:** F1-score (weighted), Accuracy

4. **Results Display**  
   Print training and test metrics, including the F1 gap to indicate potential overfitting.

5. **Return Value**  
   Return a dictionary containing test metrics and the fitted model object for comparison or saving.

In [50]:
print("🌲 Running Random Forest Regressor")
start = time.time()

# 1. Initialize
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# 2. Train
rf_model.fit(X_train, y_train.values.ravel())

# 3. Evaluate
results_table.append(evaluate_regressor(rf_model, X_train, y_train.values.ravel(), X_test, y_test.values.ravel(), "Random Forest"))
print(f"⏱️ Time: {time.time()-start:.2f}s")

🌲 Running Random Forest Regressor
✅ Random Forest Results:
   [TRAIN] MSE: 0.1388 | RMSE: 0.3726 | Accuracy: 0.9265 | F1: 0.9078 | R2: 0.1545
   [TEST] MSE: 0.1448 | RMSE: 0.3805 | Accuracy: 0.9245 | F1: 0.9052 | R2: 0.1184
   -> Gap (Overfitting): 0.0026 (F1 diff)
------------------------------------------------------------
⏱️ Time: 450.04s


### Random Forest Regressor Performance Analysis

**High Classification Performance:**  
The model achieved **92.45% Accuracy** and an **F1-Score of 0.9052** on the test set, indicating strong predictive capability for this ordinal dataset.

**Excellent Generalization:**  
The minimal gap between train and test F1-Score (**0.0026**) suggests the model **does not overfit** and generalizes well to unseen data.

**Low Prediction Error:**  
An **RMSE of 0.3805** implies that predictions are, on average, within 0.38 units of the true severity levels (1–4), which is acceptable given the rounding approach for classification metrics.

**R² Score Consideration:**  
The low **R² (0.1184)** is expected for a regression approach applied to discrete ordinal data; it does not diminish the model’s effectiveness in classification.

**Conclusion:**  
Random Forest is a **robust and reliable choice** for this task, balancing high classification accuracy with low overfitting.

In [51]:
print("🚀 Running XGBoost Regressor")
start = time.time()

# 1. Initialize with fixed parameters
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# 2. Train
xgb_model.fit(X_train, y_train)

# 3. Evaluate
results_table.append(evaluate_regressor(xgb_model, X_train, y_train, X_test, y_test, "XGBoost"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

🚀 Running XGBoost Regressor
✅ XGBoost Results:
   [TRAIN] MSE: 0.1403 | RMSE: 0.3746 | Accuracy: 0.9241 | F1: 0.9056 | R2: 0.1455
   [TEST] MSE: 0.1448 | RMSE: 0.3805 | Accuracy: 0.9230 | F1: 0.9042 | R2: 0.1182
   -> Gap (Overfitting): 0.0013 (F1 diff)
------------------------------------------------------------
⏱️ Time: 17.18s


### XGBoost Regressor Performance Analysis

**High Classification Performance:**  
XGBoost achieved **92.30% Accuracy** and an **F1-Score of 0.9042** on the test set, demonstrating strong capability in predicting ordinal severity levels.

**Minimal Overfitting:**  
The F1-Score gap between train and test is very small (**0.0013**), indicating excellent **generalization** and stable predictions.

**Low Prediction Error:**  
The **RMSE of 0.3805** suggests that the predicted values are, on average, within 0.38 units of the true severity, which is consistent with the rounding strategy for classification.

**R² Score Context:**  
The low **R² (0.1182)** is expected when using regression metrics on discrete ordinal data and does not undermine the high classification performance.

**Conclusion:**  
XGBoost is a **highly reliable model** with minimal overfitting, comparable to Random Forest, making it a strong candidate for this ordinal regression task.

In [52]:
print("⚡ Running LightGBM Regressor")
start = time.time()

# 1. Initialize
lgbm_model = LGBMRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# 2. Train
lgbm_model.fit(X_train, y_train)

# 3. Evaluate
results_table.append(evaluate_regressor(lgbm_model, X_train, y_train, X_test, y_test, "LightGBM"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

⚡ Running LightGBM Regressor
✅ LightGBM Results:
   [TRAIN] MSE: 0.1401 | RMSE: 0.3743 | Accuracy: 0.9243 | F1: 0.9058 | R2: 0.1466
   [TEST] MSE: 0.1447 | RMSE: 0.3804 | Accuracy: 0.9235 | F1: 0.9050 | R2: 0.1188
   -> Gap (Overfitting): 0.0008 (F1 diff)
------------------------------------------------------------
⏱️ Time: 57.70s


### LightGBM Regressor Performance Analysis

**Strong Classification Metrics:**  
LightGBM achieved **92.35% Accuracy** and an **F1-Score of 0.9050** on the test set, showing high effectiveness in predicting ordinal severity levels.

**Negligible Overfitting:**  
The F1-Score gap between train and test is extremely small (**0.0008**), indicating **excellent model generalization**.

**Low Prediction Error:**  
An **RMSE of 0.3804** implies predictions are, on average, within 0.38 units of the true severity, which aligns well with the rounding strategy for classification.

**R² Score Consideration:**  
The low **R² (0.1188)** is expected for regression applied to discrete ordinal values and does not diminish the high classification performance.

**Conclusion:**  
LightGBM is a **highly stable and reliable model**, offering strong predictive performance with minimal overfitting.

In [53]:
print("🐱 Running CatBoost Regressor")
start = time.time()

# 1. Initialize
cat_model = CatBoostRegressor(
    n_estimators=50,
    random_state=42,
    verbose=0
)

# 2. Train
cat_model.fit(X_train, y_train)

# 3. Evaluate
results_table.append(evaluate_regressor(cat_model, X_train, y_train, X_test, y_test, "CatBoost"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

🐱 Running CatBoost Regressor
✅ CatBoost Results:
   [TRAIN] MSE: 0.1399 | RMSE: 0.3741 | Accuracy: 0.9241 | F1: 0.9061 | R2: 0.1478
   [TEST] MSE: 0.1447 | RMSE: 0.3805 | Accuracy: 0.9229 | F1: 0.9047 | R2: 0.1184
   -> Gap (Overfitting): 0.0014 (F1 diff)
------------------------------------------------------------
⏱️ Time: 15.90s


### CatBoost Regressor Performance Analysis

**High Classification Performance:**  
CatBoost achieves **92.29% Accuracy** and an **F1-Score of 0.9047** on the test set, demonstrating reliable predictions for ordinal severity levels.

**Minimal Overfitting:**  
The F1-Score gap between train and test is very small (**0.0014**), indicating the model **generalizes well** without overfitting.

**Prediction Error:**  
With an **RMSE of 0.3805**, predictions are on average within 0.38 units of the actual severity, supporting the rounding approach for classification.

**R² Score Note:**  
The low **R² (0.1184)** is expected for regression on discrete ordinal values and does not undermine the strong classification metrics.

**Conclusion:**  
CatBoost is a **robust and stable choice**, providing consistent performance and minimal overfitting in this ordinal regression-to-classification setup.

In [54]:
# 1. Create comparison DataFrame
df_results = pd.DataFrame(results_table)

# Sort by F1-Score (highest first)
df_results = df_results.sort_values(by='F1', ascending=False)

display_cols = ['Model', 'RMSE', 'MSE', 'F1', 'Accuracy', 'R2']

print("\n🏆 MODEL LEADERBOARD")
print(df_results[display_cols].to_markdown(index=False))


# 2. Select the best model
best_model_info = df_results.iloc[0]
best_model_name = best_model_info['Model']
best_model_object = best_model_info['Object']

print(f"\n🥇 WINNER: {best_model_name}")
print(f"   with F1-Score: {best_model_info['F1']:.4f}, RMSE: {best_model_info['RMSE']:.4f}, MSE: {best_model_info['MSE']:.4f}, R2: {best_model_info['R2']:.4f}, Accuracy : {best_model_info['Accuracy']:.4f}")

# 3. Save the best model
filename = f'best_model_{best_model_name.replace(' ', '_')}.pkl'
joblib.dump(best_model_object, filename)
print(f"💾 Saved the best model to file: {filename}")


🏆 MODEL LEADERBOARD
| Model         |     RMSE |      MSE |       F1 |   Accuracy |       R2 |
|:--------------|---------:|---------:|---------:|-----------:|---------:|
| Random Forest | 0.380463 | 0.144752 | 0.905237 |   0.924465 | 0.11838  |
| LightGBM      | 0.380366 | 0.144679 | 0.905035 |   0.923462 | 0.11883  |
| CatBoost      | 0.380457 | 0.144748 | 0.904708 |   0.922929 | 0.118408 |
| XGBoost       | 0.380501 | 0.144781 | 0.904238 |   0.92304  | 0.118207 |

🥇 WINNER: Random Forest
   with F1-Score: 0.9052, RMSE: 0.3805, MSE: 0.1448, R2: 0.1184, Accuracy : 0.9245
💾 Saved the best model to file: best_model_Random_Forest.pkl


# Model Selection Analysis

**Selection Criteria:**  
The **F1-Score** was prioritized over Accuracy as the primary metric for selecting the best model. This ensures the evaluation focuses on the model's ability to correctly classify all severity levels, especially the minority classes, which Accuracy alone can often misrepresent in imbalanced datasets.

**Performance:**  
Random Forest emerged as the superior model, achieving the **highest F1-Score**. This indicates it provides the best balance between precision and recall, making it the most reliable choice for detecting critical accident severities.

**Conclusion:**  
Random Forest is selected as the **final model** specifically due to its top-ranking **F1-Score performance**.